# Figure S1A — Adrenal Insufficiency Positives Binary Timeline (Note Level)

Shows note-level LLM binary predictions for patients confirmed positive in the gold standard.  
Each note is rendered as a fixed-width tick (10 days) centered on the note date.  
X-axis is days relative to gold-standard AE onset (t = 0, dashed line).  
Color: **red** = predicted positive (prob > 0.810), **blue** = predicted negative.

**Data sources** (under `figures/figures_data/figure 1/data`):
- `noteleveldata/11_patients_results.csv` — note-level probability predictions (thresholded to binary)
- `2026May01_merged_ae_with_apr_full.csv` — old gold standard with AE start dates
- `final_gold_standard_1k.csv` — new gold standard (patient-level 0/1)

Outputs are written to `figure 1/results/supp/`.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib

%matplotlib inline

matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")


In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS — notebook lives in figure 1/scripts/
#   figures/
#   ├── figures_data/figure 1/data/   ← inputs (shared OneDrive data dir)
#   └── v1/figure 1/
#       ├── scripts/                  ← this notebook
#       └── results/supp/             ← output PDF + CSV
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

LLM_PATH    = DATA / "noteleveldata" / "11_patients_results.csv"
OLD_GS_PATH = DATA / "2026May01_merged_ae_with_apr_full.csv"
NEW_GS_PATH = DATA / "final_gold_standard_1k.csv"
OUT_PATH    = RESULTS / "supp" / "Positive_Binary_Adrenal_S1A.pdf"

for p in [LLM_PATH, OLD_GS_PATH, NEW_GS_PATH]:
    assert p.exists(), f"Missing: {p}"
print("All input files found.")
print(f"Data: {DATA}")
print(f"Results: {RESULTS / 'supp'}")


In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
LLM_AE_COL  = "adrenal insufficiency"
NEW_GS_COL  = "adrenal_insufficiency"
OLD_GS_TOX  = "adrenal insufficiency"
COLOR_POS   = "#8b0000"
COLOR_NEG   = "#1a3a6e"
TICK_HEIGHT = 0.55
TICK_WIDTH  = 10          # fixed display width per note (days)
FONT        = "Arial"
FIG_W       = 260 / 72    # width in inches
FIG_H       = 140 / 72    # height in inches
X_ABS_SHARED = 1800

AE_THRESHOLDS = {
    "pneumonitis":           0.710,
    "adrenal insufficiency": 0.810,
    "adrenal_insufficiency": 0.810,
    "liver toxicity":        0.010,
    "liver_toxicity":        0.010,
    "colitis":               0.710,
    "hyperthyroidism":       0.810,
    "hypothyroidism":        0.510,
}
THRESHOLD = AE_THRESHOLDS.get(LLM_AE_COL, 0.5)
print(f"Threshold for {LLM_AE_COL}: {THRESHOLD}")


In [ ]:
def standardize_mrn(series: pd.Series) -> pd.Series:
    """Strip MRN prefixes and extract numeric portion — no zero-padding."""
    return (
        series.astype(str)
        .str.replace(r"[Pp]-|[Mm][Ss][Kk]-", "", regex=True)
        .str.extract(r"(\d+)")[0]
        .str.lstrip("0")
    )

def get_mrn_col(df: pd.DataFrame, path: str) -> str:
    """Return the MRN column name using case-insensitive lookup."""
    col = next((c for c in df.columns if c.lower() == "mrn"), None)
    if col is None:
        raise ValueError(f"No MRN column found in {path}. Columns: {list(df.columns)}")
    return col


In [ ]:
# Do not load free-text rationale (may contain clinical-note excerpts).
df = pd.read_csv(
    LLM_PATH,
    usecols=lambda c: c.lower() == "mrn" or c in ("window_start", LLM_AE_COL),
)
print(f"Read {len(df):,} rows from {LLM_PATH.name}")

In [ ]:
def load_llm(path, df):
    """Load note-level predictions; threshold probability to binary call."""
    df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
    df["mrn_std"]      = standardize_mrn(df[get_mrn_col(df, str(path))])
    df["prob"]         = pd.to_numeric(df[LLM_AE_COL], errors="coerce").clip(0, 1)
    df[LLM_AE_COL]    = (df["prob"] > THRESHOLD).astype(int)
    return df.dropna(subset=["mrn_std", "window_start", "prob"])

llm = load_llm(LLM_PATH, df)
print(f"Loaded {len(llm)} notes for {llm['mrn_std'].nunique()} patients")
print(f"Positive notes: {(llm[LLM_AE_COL] == 1).sum()}, Negative: {(llm[LLM_AE_COL] == 0).sum()}")

In [ ]:
def load_gold_standard(old_gs_path):
    """Return one row per confirmed-positive patient with earliest AE start date."""
    old_gs = pd.read_csv(old_gs_path, low_memory=False)
    old_gs["Toxicity"] = old_gs["Toxicity"].str.strip().str.lower()
    old_gs = old_gs[old_gs["Toxicity"] == OLD_GS_TOX.lower()].copy()
    old_gs["Start Date"] = pd.to_datetime(old_gs["Start Date"], errors="coerce")
    old_gs = old_gs.dropna(subset=["MRN", "Start Date"])
    old_gs["mrn_std"] = standardize_mrn(old_gs["MRN"])
    return (
        old_gs.sort_values("Start Date")
        .groupby("mrn_std", as_index=False)["Start Date"]
        .first()
        .rename(columns={"Start Date": "ae_start"})
    )

gs = load_gold_standard(OLD_GS_PATH)
print(f"Gold standard: {len(gs)} positive patients with AE dates")

In [ ]:
def build_plot_data(llm, gs):
    """Inner-join notes with GS; compute note position relative to AE onset."""
    merged = llm.merge(gs, on="mrn_std", how="inner")
    merged["days_rel"]       = (merged["window_start"] - merged["ae_start"]).dt.total_seconds() / 86400
    merged["days_rel_start"] = merged["days_rel"] - TICK_WIDTH / 2
    merged["days_rel_end"]   = merged["days_rel"] + TICK_WIDTH / 2
    return merged

plot_df = build_plot_data(llm, gs)
print(f"Plot data: {len(plot_df)} notes across {plot_df['mrn_std'].nunique()} matched patients")


In [ ]:
def get_asterisk_mrns(plot_df, old_gs_positive_mrns, new_gs_path):
    """Return MRNs whose LLM call disagrees with old GS but agrees with new GS."""
    llm_call = (plot_df.groupby("mrn_std")[LLM_AE_COL].max() == 1).astype(int)
    old_gs_label = pd.Series(
        [1 if m in old_gs_positive_mrns else 0 for m in llm_call.index],
        index=llm_call.index
    )
    new_gs = pd.read_csv(new_gs_path)
    new_gs["mrn_std"] = standardize_mrn(new_gs[get_mrn_col(new_gs, str(new_gs_path))])
    new_gs_map = new_gs.set_index("mrn_std")[NEW_GS_COL].to_dict()
    new_gs_label = pd.Series(
        [new_gs_map.get(m, np.nan) for m in llm_call.index],
        index=llm_call.index, dtype=float
    )
    mask = (llm_call != old_gs_label) & (llm_call == new_gs_label)
    return set(llm_call.index[mask])

old_gs_positive_mrns = set(gs["mrn_std"])
asterisk_mrns = get_asterisk_mrns(plot_df, old_gs_positive_mrns, NEW_GS_PATH)
print(f"Asterisk patients: {len(asterisk_mrns)}")


In [ ]:
def build_index(plot_df):
    """Canonical patient index: sort GS-joined MRNs by integer value ascending."""
    return (
        plot_df.drop_duplicates("mrn_std")[["mrn_std"]]
        .assign(mrn_int=lambda d: pd.to_numeric(d["mrn_std"], errors="coerce"))
        .sort_values("mrn_int", ascending=True)
        .reset_index(drop=True)
        .assign(pt_idx=lambda d: d.index + 1)
    )

span = build_index(plot_df)

In [ ]:
# --- Merge patient index ---
plot_df = plot_df.merge(span[["mrn_std", "pt_idx"]], on="mrn_std")
n_patients = span.shape[0]
mrn_to_ptidx = span.set_index("mrn_std")["pt_idx"].to_dict()

x_abs = X_ABS_SHARED

# --- Figure ---
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

for _, row in plot_df.iterrows():
    color = COLOR_POS if int(row[LLM_AE_COL]) == 1 else COLOR_NEG
    ax.barh(
        y=row["pt_idx"], width=TICK_WIDTH,
        left=row["days_rel_start"], height=TICK_HEIGHT,
        color=color, linewidth=0,
    )

# Dashed line at AE onset (t = 0)
ax.axvline(x=0, color="black", linestyle="--", linewidth=0.6, zorder=5)

# Asterisks
x_offset = x_abs * 0.01
rightmost_x = plot_df.groupby("mrn_std")["days_rel_end"].max()
for mrn in asterisk_mrns:
    if mrn in mrn_to_ptidx and mrn in rightmost_x.index:
        ax.text(
            rightmost_x[mrn] + x_offset, mrn_to_ptidx[mrn], "*",
            va="center", ha="left", fontsize=5, color="black",
            fontfamily=FONT, fontweight="bold"
        )

# Axes
ax.set_xlim(-x_abs * 1.05, x_abs * 1.05)
ax.set_xticks(np.arange(-1500, 1501, 500))
ytick_vals = [i for i in range(1, n_patients + 1) if i % 2 == 0]
ax.set_yticks(ytick_vals)
ax.set_yticklabels(ytick_vals, fontsize=6, fontfamily=FONT)
ax.set_ylim(0.3, n_patients + 1.0)
ax.set_ylabel("Patient Index", fontsize=7, fontfamily=FONT)
ax.set_xlabel("Days relative to AE start", fontsize=7, fontfamily=FONT)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_linewidth(0.6)
ax.tick_params(axis="both", which="both", length=2, width=0.6, labelsize=6)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")

# Legend outside plot
legend_handles = [
    mpatches.Patch(color=COLOR_POS, label="Predicted positive"),
    mpatches.Patch(color=COLOR_NEG, label="Predicted negative"),
]
ax.legend(handles=legend_handles, fontsize=5, frameon=False,
          loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0,
          prop={"family": FONT, "size": 5})

plt.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=450, facecolor="white")
print(f"Saved: {OUT_PATH}")
plt.show()

In [ ]:
# ===========================================================================
# PLOT 1 — Binary predicted positive / negative
# ===========================================================================

# ---------------------------------------------------------------------------
# Patient index
# ---------------------------------------------------------------------------

# Prevent duplicate pt_idx columns if this cell is accidentally re-run
if "pt_idx" in plot_df.columns:
    plot_df = plot_df.drop(columns=["pt_idx"])

plot_df = plot_df.merge(
    span[["mrn_std", "pt_idx"]],
    on="mrn_std",
    how="left"
)

n_patients = span.shape[0]

mrn_to_ptidx = (
    span
    .set_index("mrn_std")["pt_idx"]
    .to_dict()
)

x_abs = X_ABS_SHARED


# ---------------------------------------------------------------------------
# Figure dimensions
# ---------------------------------------------------------------------------

FIG_W_PT = 340
FIG_H_PT = 140

FIG_W = FIG_W_PT / 72
FIG_H = FIG_H_PT / 72

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    facecolor="white"
)


# ---------------------------------------------------------------------------
# FIXED AXES POSITION
#
# These exact values MUST also be used for Plot 2.
#
# left   = 36 pt
# bottom = 31 pt
# width  = 190 pt
# height = 99 pt
# ---------------------------------------------------------------------------

AX_LEFT_PT = 36
AX_BOTTOM_PT = 31
AX_WIDTH_PT = 190
AX_HEIGHT_PT = 99

ax = fig.add_axes([
    AX_LEFT_PT / FIG_W_PT,
    AX_BOTTOM_PT / FIG_H_PT,
    AX_WIDTH_PT / FIG_W_PT,
    AX_HEIGHT_PT / FIG_H_PT
])


# ---------------------------------------------------------------------------
# Plot notes
# ---------------------------------------------------------------------------

for _, row in plot_df.iterrows():

    color = (
        COLOR_POS
        if int(row[LLM_AE_COL]) == 1
        else COLOR_NEG
    )

    ax.barh(
        y=row["pt_idx"],
        width=TICK_WIDTH,
        left=row["days_rel_start"],
        height=TICK_HEIGHT,
        color=color,
        linewidth=0,
    )


# ---------------------------------------------------------------------------
# AE onset
# ---------------------------------------------------------------------------

ax.axvline(
    x=0,
    color="black",
    linestyle="--",
    linewidth=0.6,
    zorder=5
)


# ---------------------------------------------------------------------------
# Asterisks
# ---------------------------------------------------------------------------

x_offset = x_abs * 0.01

rightmost_x = (
    plot_df
    .groupby("mrn_std")["days_rel_end"]
    .max()
)

for mrn in asterisk_mrns:

    if (
        mrn in mrn_to_ptidx
        and mrn in rightmost_x.index
    ):

        ax.text(
            rightmost_x[mrn] + x_offset,
            mrn_to_ptidx[mrn],
            "*",
            va="center",
            ha="left",
            fontsize=5,
            color="black",
            fontfamily=FONT,
            fontweight="bold",
        )


# ---------------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------------

ax.set_xlim(
    -x_abs * 1.05,
    x_abs * 1.05
)

ax.set_xticks(
    np.arange(-1500, 1501, 500)
)

ytick_vals = [
    i
    for i in range(1, n_patients + 1)
    if i % 2 == 0
]

ax.set_yticks(ytick_vals)

ax.set_yticklabels(
    ytick_vals,
    fontsize=6,
    fontfamily=FONT
)

ax.set_ylim(
    0.3,
    n_patients + 1.0
)

ax.set_ylabel(
    "Patient Index",
    fontsize=7,
    fontfamily=FONT
)

ax.set_xlabel(
    "Days relative to AE start",
    fontsize=7,
    fontfamily=FONT
)


# ---------------------------------------------------------------------------
# Styling
# ---------------------------------------------------------------------------

ax.spines[["top", "right"]].set_visible(False)

ax.spines[["left", "bottom"]].set_linewidth(0.6)

ax.tick_params(
    axis="both",
    which="both",
    length=2,
    width=0.6,
    labelsize=6
)

ax.set_facecolor("white")

fig.patch.set_facecolor("white")


# ---------------------------------------------------------------------------
# Legend
# ---------------------------------------------------------------------------

legend_handles = [
    mpatches.Patch(
        color=COLOR_POS,
        label="Predicted positive"
    ),
    mpatches.Patch(
        color=COLOR_NEG,
        label="Predicted negative"
    ),
]

ax.legend(
    handles=legend_handles,
    fontsize=5,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.06, 1.0),
    borderaxespad=0,
    prop={
        "family": FONT,
        "size": 5
    }
)


# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------

# IMPORTANT:
# Do NOT use tight_layout().
# It would change the axes position and break alignment with Plot 2.

fig.savefig(
    OUT_PATH,
    dpi=450,
    facecolor="white"
)

print(f"Saved: {OUT_PATH}")

plt.show()

In [ ]:
# ===========================================================================
# PLOT 1 — Binary predicted positive / negative
# ===========================================================================

# ---------------------------------------------------------------------------
# Patient index
# ---------------------------------------------------------------------------

# Prevent duplicate pt_idx columns if this cell is re-run
if "pt_idx" not in plot_df.columns:
    plot_df = plot_df.merge(
        span[["mrn_std", "pt_idx"]],
        on="mrn_std",
        how="left"
    )

n_patients = span.shape[0]

mrn_to_ptidx = (
    span
    .set_index("mrn_std")["pt_idx"]
    .to_dict()
)

x_abs = X_ABS_SHARED


# ===========================================================================
# FIGURE GEOMETRY
#
# THESE VALUES MUST BE IDENTICAL IN PLOT 2
# Figure resized to 267.3875pt x 131.8697pt; axes geometry scaled
# proportionally (Rw=0.876680, Rh=0.941926) from the 305x140 original
# so the plot area keeps the same relative alignment as before.
# ===========================================================================

FIG_W_PT = 308.8382
FIG_H_PT = 131.8697

FIG_W = FIG_W_PT / 72
FIG_H = FIG_H_PT / 72


# ---------------------------------------------------------------------------
# Create figure
# ---------------------------------------------------------------------------

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    facecolor="white"
)


# ===========================================================================
# MAIN PLOTTING AXES
#
# These are ABSOLUTE POINT positions.
# Therefore the graph itself will be exactly the same physical size/location
# in both PDFs.
# ===========================================================================

AX_LEFT_PT = 31.5605
AX_BOTTOM_PT = 29.1997
AX_WIDTH_PT = 208.02
AX_HEIGHT_PT = 93.2507

ax = fig.add_axes([
    AX_LEFT_PT / FIG_W_PT,
    AX_BOTTOM_PT / FIG_H_PT,
    AX_WIDTH_PT / FIG_W_PT,
    AX_HEIGHT_PT / FIG_H_PT
])


# ===========================================================================
# Plot notes
# ===========================================================================

for _, row in plot_df.iterrows():

    color = (
        COLOR_POS
        if int(row[LLM_AE_COL]) == 1
        else COLOR_NEG
    )

    ax.barh(
        y=row["pt_idx"],
        width=TICK_WIDTH,
        left=row["days_rel_start"],
        height=TICK_HEIGHT,
        color=color,
        linewidth=0
    )


# ===========================================================================
# AE onset
# ===========================================================================

ax.axvline(
    x=0,
    color="black",
    linestyle="--",
    linewidth=0.6,
    zorder=5
)


# ===========================================================================
# Asterisks
# ===========================================================================

x_offset = x_abs * 0.01

rightmost_x = (
    plot_df
    .groupby("mrn_std")["days_rel_end"]
    .max()
)

for mrn in asterisk_mrns:

    if (
        mrn in mrn_to_ptidx
        and mrn in rightmost_x.index
    ):

        ax.text(
            rightmost_x[mrn] + x_offset,
            mrn_to_ptidx[mrn],
            "*",
            va="center",
            ha="left",
            fontsize=5,
            color="black",
            fontfamily=FONT,
            fontweight="bold"
        )


# ===========================================================================
# AXES
# ===========================================================================

ax.set_xlim(
    -x_abs * 1.05,
    x_abs * 1.05
)

ax.set_xticks(
    np.arange(-1500, 1501, 500)
)

ytick_vals = [
    i
    for i in range(1, n_patients + 1)
    if i % 2 == 0
]

ax.set_yticks(ytick_vals)

ax.set_yticklabels(
    ytick_vals,
    fontsize=6,
    fontfamily=FONT
)

ax.set_ylim(
    0.3,
    n_patients + 1.0
)

ax.set_ylabel(
    "Patient Index",
    fontsize=7,
    fontfamily=FONT
)

ax.set_xlabel(
    "Days relative to AE start",
    fontsize=7,
    fontfamily=FONT
)


# ===========================================================================
# STYLING
# ===========================================================================

ax.spines[["top", "right"]].set_visible(False)

ax.spines[["left", "bottom"]].set_linewidth(0.6)

ax.tick_params(
    axis="both",
    which="both",
    length=2,
    width=0.6,
    labelsize=6
)

ax.set_facecolor("white")

fig.patch.set_facecolor("white")


# ===========================================================================
# LEGEND
#
# Positioned using FIGURE coordinates rather than ax.transAxes.
# This guarantees it stays inside the PDF and doesn't change the plot axes.
# ===========================================================================

legend_handles = [
    mpatches.Patch(
        color=COLOR_POS,
        label="Predicted positive"
    ),
    mpatches.Patch(
        color=COLOR_NEG,
        label="Predicted negative"
    )
]

# Right edge of graph = 31.5605 + 208.02 = 239.5805 pt
#
# Legend begins at 243.9639 pt.
# This is roughly a 4.4 pt gap from the graph.
#
# Top of graph = 29.1997 + 93.2507 = 122.4504 pt.

fig.legend(
    handles=legend_handles,
    loc="upper left",
    bbox_to_anchor=(
        243.9639 / FIG_W_PT,
        122.4504 / FIG_H_PT
    ),
    frameon=False,
    borderaxespad=0,
    prop={
        "family": FONT,
        "size": 5
    }
)


# ===========================================================================
# SAVE
# ===========================================================================

fig.savefig(
    OUT_PATH,
    dpi=450,
    facecolor="white"
)

print(f"Saved: {OUT_PATH}")

plt.show()

In [ ]:
# ===========================================================================
# EXPORT — underlying data for this panel (de-identified)
# ===========================================================================

CSV_OUT = OUT_PATH.with_name(OUT_PATH.stem + "_results.csv")

export_df = (
    plot_df[["mrn_std", "pt_idx", "days_rel", "prob", LLM_AE_COL]]
    .rename(columns={
        "days_rel": "days_relative_to_ae_start",
        "prob": "llm_probability",
        LLM_AE_COL: "llm_binary_call",
    })
    .sort_values(["pt_idx", "days_relative_to_ae_start"])
    .reset_index(drop=True)
)

export_df["asterisk_patient"] = export_df["mrn_std"].isin(asterisk_mrns).astype(int)
export_df["threshold"] = THRESHOLD
export_df["toxicity"] = LLM_AE_COL

# Drop MRN — pt_idx is the de-identified patient key used by the figure
export_df = export_df.drop(columns=["mrn_std"])

assert not any(c.lower() in ("mrn", "mrn_std") for c in export_df.columns), "MRN leaked"
assert not any("date" in c.lower() for c in export_df.columns), "date column leaked"

export_df.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT.name}")
print(f"  {len(export_df):,} notes, {export_df['pt_idx'].nunique()} patients, "
      f"{int(export_df['llm_binary_call'].sum()):,} positive calls")